In [0]:
# =============================================================================
# NSSC 2026 EDITS - NON-GOVERNMENT SCHOOLS
# Complete production version with full cell-by-cell validation
# =============================================================================

from functools import reduce
import re
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

# =============================================================================
# 1. CONFIGURATION
# =============================================================================
CENSUS_YEAR = 2026
TABLE_1_SOURCE = "pr_dap_efs.business_sdpt_abs.nssc_ngs_table_1_2026"
TABLE_2_SOURCE = "pr_dap_efs.business_sdpt_abs.nssc_ngs_table_2_2026"
OUTPUT_TABLE = "pr_dap_efs.business_sdpt_abs.nssc_ngs_edits_2026"
OUTPUT_DECIMAL_TYPE = DecimalType(24, 1)
TOLERANCE = 0.0001

JURISDICTIONS = ["NSW", "VIC", "QLD", "SA", "WA", "TAS", "NT", "ACT"]
AFFILIATIONS = ["Catholic", "Independent"]

VALID_TABLE_2_GRADES = [1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 15, 16]
PRIMARY_TABLE_2_GRADES = [1, 2, 3, 4, 5, 6, 7, 9]
SECONDARY_TABLE_2_GRADES = [10, 11, 12, 13, 14, 15, 16]

SCHOOL_COUNT_UNIT = "School Count"
STUDENT_COUNT_UNIT = "Student Count (including students at special schools)"
STUDENT_FTE_UNIT = "Student FTE (including students at special schools)"

# =============================================================================
# 2. HELPERS
# =============================================================================
def union_all(dataframes):
    if not dataframes:
        raise ValueError("union_all requires at least one DataFrame.")
    return reduce(lambda left, right: left.unionByName(right, allowMissingColumns=True), dataframes)


def add_columns(expressions):
    return reduce(lambda left, right: left + right, expressions) if expressions else F.lit(0)


def add_edits_dimensions(dataframe):
    return (
        dataframe
        .withColumn(
            "State_Territory",
            F.when(F.col("state") == 1, "NSW")
             .when(F.col("state") == 2, "VIC")
             .when(F.col("state") == 3, "QLD")
             .when(F.col("state") == 4, "SA")
             .when(F.col("state") == 5, "WA")
             .when(F.col("state") == 6, "TAS")
             .when(F.col("state") == 7, "NT")
             .when(F.col("state") == 8, "ACT")
        )
        .withColumn(
            "School_Affiliation",
            F.when(F.col("affiliation") == 3, "Catholic")
             .when(F.col("affiliation").isin(2, 4), "Independent")
        )
    )


def create_long_group(dataframe, counting_unit, classification, category_column, value_column, value_alias="Value"):
    return (
        dataframe
        .filter(F.col(category_column).isNotNull())
        .groupBy(
            "State_Territory",
            "School_Affiliation",
            F.col(category_column).alias("Category")
        )
        .agg(F.sum(F.col(value_column)).alias(value_alias))
        .withColumn("Counting_Unit", F.lit(counting_unit))
        .withColumn("Classifications", F.lit(classification))
        .select(
            "Counting_Unit", "Classifications", "Category",
            "State_Territory", "School_Affiliation", value_alias
        )
    )


def apply_output_format(dataframe):
    result = dataframe
    for column_name in MEASURE_COLUMNS:
        result = result.withColumn(
            column_name,
            F.when(
                F.col("Counting_Unit") == STUDENT_FTE_UNIT,
                F.round(F.col(column_name), 1).cast(OUTPUT_DECIMAL_TYPE)
            ).otherwise(
                F.round(F.col(column_name), 0).cast(OUTPUT_DECIMAL_TYPE)
            )
        )
    return result


def add_match_columns(dataframe, expected_column="Expected_Value", actual_column="Actual_Value"):
    return (
        dataframe
        .withColumn("Difference", F.col(actual_column) - F.col(expected_column))
        .withColumn("Tolerance", F.lit(TOLERANCE))
        .withColumn(
            "Status",
            F.when(F.abs(F.col("Difference")) < F.col("Tolerance"), "MATCH").otherwise("CHECK")
        )
    )

# =============================================================================
# 3. CHECK AND LOAD SOURCES
# =============================================================================
for source_name, source_table in [("Table 1", TABLE_1_SOURCE), ("Table 2", TABLE_2_SOURCE)]:
    if not spark.catalog.tableExists(source_table):
        raise ValueError(f"{source_name} does not exist: {source_table}")

table_1 = spark.table(TABLE_1_SOURCE).filter(F.col("year") == CENSUS_YEAR)
table_2 = spark.table(TABLE_2_SOURCE).filter(F.col("year") == CENSUS_YEAR)

table_1_row_count = table_1.count()
table_2_row_count = table_2.count()
if table_1_row_count == 0:
    raise ValueError(f"Table 1 contains no records for {CENSUS_YEAR}.")
if table_2_row_count == 0:
    raise ValueError(f"Table 2 contains no records for {CENSUS_YEAR}.")

table_1_standard = add_edits_dimensions(table_1)
table_2_standard = (
    add_edits_dimensions(table_2)
    .withColumn(
        "Grade_Code",
        F.expr("try_cast(NULLIF(TRIM(CAST(grade AS STRING)), '') AS INT)")
    )
)

# =============================================================================
# 4. SOURCE VALIDATION
# =============================================================================
table_1_dimension_issues = table_1_standard.filter(
    F.col("State_Territory").isNull()
    | F.col("School_Affiliation").isNull()
    | F.col("schooltype").isNull()
    | ~F.col("schooltype").isin(1, 2, 3, 4)
    | F.col("school_count").isNull()
    | (F.col("school_count") < 0)
)

if table_1_dimension_issues.count() > 0:
    display(table_1_dimension_issues)
    raise ValueError("Table 1 contains invalid state, affiliation, school type or school count values.")

table_2_dimension_issues = table_2_standard.filter(
    F.col("State_Territory").isNull()
    | F.col("School_Affiliation").isNull()
    | F.col("Grade_Code").isNull()
    | ~F.col("Grade_Code").isin(VALID_TABLE_2_GRADES)
    | F.col("stud_typ").isNull()
    | F.col("sex").isNull()
    | F.col("fp_ind").isNull()
    | F.col("enrolments").isNull()
    | (F.col("enrolments") < 0)
    | F.col("fte").isNull()
    | (F.col("fte") < 0)
)

if table_2_dimension_issues.count() > 0:
    display(table_2_dimension_issues)
    raise ValueError(
        "Table 2 contains invalid dimensions or measures. Valid grade codes are "
        "1-7, 9, and 10-16; code 8 is invalid."
    )

# =============================================================================
# 5. CLASSIFY SOURCE RECORDS
# =============================================================================
table_1_classified = table_1_standard.withColumn(
    "School_Level_Category",
    F.when(F.col("schooltype") == 1, "Primary")
     .when(F.col("schooltype") == 2, "Secondary")
     .when(F.col("schooltype") == 3, "Combined")
     .when(F.col("schooltype") == 4, "Special")
)

table_2_classified = (
    table_2_standard
    .withColumn(
        "Sex_Category",
        F.when(F.col("sex") == 1, "Male")
         .when(F.col("sex") == 2, "Female")
         .when(F.col("sex") == 3, "Other")
         .when(F.col("sex") == 9, "Not Stated/Inadequately described")
    )
    .withColumn(
        "Indigenous_Category",
        F.when(F.col("stud_typ") == 4, "Aboriginal and/or Torres Strait Islander")
         .when(F.col("stud_typ") == 0, "Non-Indigenous")
         .when(F.col("stud_typ") == 11, "Not-stated")
    )
    .withColumn("attendance_text", F.lower(F.trim(F.col("fp_ind").cast("string"))))
    .withColumn(
        "Attendance_Category",
        F.when(F.col("attendance_text").isin("full time", "full-time", "fulltime"), "Full-time")
         .when(F.col("attendance_text").isin("part time", "part-time", "parttime"), "Part-time")
    )
    .withColumn(
        "Student_School_Level",
        F.when(F.col("Grade_Code").isin(PRIMARY_TABLE_2_GRADES), "Primary")
         .when(F.col("Grade_Code").isin(SECONDARY_TABLE_2_GRADES), "Secondary")
    )
)

table_2_category_issues = table_2_classified.filter(
    F.col("Sex_Category").isNull()
    | F.col("Indigenous_Category").isNull()
    | F.col("Attendance_Category").isNull()
    | F.col("Student_School_Level").isNull()
)
if table_2_category_issues.count() > 0:
    display(table_2_category_issues)
    raise ValueError("Table 2 contains unmapped sex, Indigenous status, attendance or grade values.")

# =============================================================================
# 6. CREATE LONG-FORM SOURCE VALUES
# =============================================================================
school_count_long = create_long_group(
    table_1_classified, SCHOOL_COUNT_UNIT, "School Level",
    "School_Level_Category", "school_count"
)

student_count_sex_long = create_long_group(
    table_2_classified, STUDENT_COUNT_UNIT, "Sex", "Sex_Category", "enrolments"
)
student_count_indigenous_long = create_long_group(
    table_2_classified, STUDENT_COUNT_UNIT, "Indigenous Status", "Indigenous_Category", "enrolments"
)
student_count_attendance_long = create_long_group(
    table_2_classified, STUDENT_COUNT_UNIT, "Attendance Status", "Attendance_Category", "enrolments"
)
student_count_school_level_long = create_long_group(
    table_2_classified, STUDENT_COUNT_UNIT, "School Level", "Student_School_Level", "enrolments"
)
student_fte_sex_long = create_long_group(
    table_2_classified, STUDENT_FTE_UNIT, "Sex", "Sex_Category", "fte"
)
student_fte_indigenous_long = create_long_group(
    table_2_classified, STUDENT_FTE_UNIT, "Indigenous Status", "Indigenous_Category", "fte"
)
student_fte_attendance_long = create_long_group(
    table_2_classified, STUDENT_FTE_UNIT, "Attendance Status", "Attendance_Category", "fte"
)
student_fte_school_level_long = create_long_group(
    table_2_classified, STUDENT_FTE_UNIT, "School Level", "Student_School_Level", "fte"
)

all_long_values = union_all([
    school_count_long,
    student_count_sex_long,
    student_count_indigenous_long,
    student_count_attendance_long,
    student_count_school_level_long,
    student_fte_sex_long,
    student_fte_indigenous_long,
    student_fte_attendance_long,
    student_fte_school_level_long
])

# =============================================================================
# 7. DEFINE REQUIRED OUTPUT ROWS
# =============================================================================
row_order = [
    (SCHOOL_COUNT_UNIT, "School Level", "Primary"),
    (SCHOOL_COUNT_UNIT, "School Level", "Secondary"),
    (SCHOOL_COUNT_UNIT, "School Level", "Combined"),
    (SCHOOL_COUNT_UNIT, "School Level", "Special"),
    (SCHOOL_COUNT_UNIT, "School Level", "Total"),
    (STUDENT_COUNT_UNIT, "Sex", "Male"),
    (STUDENT_COUNT_UNIT, "Sex", "Female"),
    (STUDENT_COUNT_UNIT, "Sex", "Other"),
    (STUDENT_COUNT_UNIT, "Sex", "Not Stated/Inadequately described"),
    (STUDENT_COUNT_UNIT, "Sex", "Total"),
    (STUDENT_COUNT_UNIT, "Indigenous Status", "Aboriginal and/or Torres Strait Islander"),
    (STUDENT_COUNT_UNIT, "Indigenous Status", "Non-Indigenous"),
    (STUDENT_COUNT_UNIT, "Indigenous Status", "Not-stated"),
    (STUDENT_COUNT_UNIT, "Indigenous Status", "Total"),
    (STUDENT_COUNT_UNIT, "Attendance Status", "Full-time"),
    (STUDENT_COUNT_UNIT, "Attendance Status", "Part-time"),
    (STUDENT_COUNT_UNIT, "Attendance Status", "Total"),
    (STUDENT_COUNT_UNIT, "School Level", "Primary"),
    (STUDENT_COUNT_UNIT, "School Level", "Secondary"),
    (STUDENT_COUNT_UNIT, "School Level", "Total"),
    (STUDENT_FTE_UNIT, "Sex", "Male"),
    (STUDENT_FTE_UNIT, "Sex", "Female"),
    (STUDENT_FTE_UNIT, "Sex", "Other"),
    (STUDENT_FTE_UNIT, "Sex", "Not Stated/Inadequately described"),
    (STUDENT_FTE_UNIT, "Sex", "Total"),
    (STUDENT_FTE_UNIT, "Indigenous Status", "Aboriginal and/or Torres Strait Islander"),
    (STUDENT_FTE_UNIT, "Indigenous Status", "Non-Indigenous"),
    (STUDENT_FTE_UNIT, "Indigenous Status", "Not-stated"),
    (STUDENT_FTE_UNIT, "Indigenous Status", "Total"),
    (STUDENT_FTE_UNIT, "Attendance Status", "Full-time"),
    (STUDENT_FTE_UNIT, "Attendance Status", "Part-time"),
    (STUDENT_FTE_UNIT, "Attendance Status", "Total"),
    (STUDENT_FTE_UNIT, "School Level", "Primary"),
    (STUDENT_FTE_UNIT, "School Level", "Secondary"),
    (STUDENT_FTE_UNIT, "School Level", "Total")
]

row_order_df = spark.createDataFrame(
    [(i, cu, cl, ca) for i, (cu, cl, ca) in enumerate(row_order, start=1)],
    ["_sort_order", "Counting_Unit", "Classifications", "Categories"]
)

category_scaffold = (
    row_order_df
    .filter(F.col("Categories") != "Total")
    .select("_sort_order", "Counting_Unit", "Classifications", "Categories")
)

# =============================================================================
# 8. CREATE THE UNFORMATTED OUTPUT
# =============================================================================
pivot_aggregations = [
    F.sum(
        F.when(
            (F.col("State_Territory") == jurisdiction)
            & (F.col("School_Affiliation") == affiliation),
            F.col("Value")
        ).otherwise(F.lit(0))
    ).alias(f"{jurisdiction}_{affiliation}")
    for jurisdiction in JURISDICTIONS
    for affiliation in AFFILIATIONS
]

category_values = (
    all_long_values
    .groupBy("Counting_Unit", "Classifications", F.col("Category").alias("Categories"))
    .agg(*pivot_aggregations)
)

category_output = (
    category_scaffold
    .join(category_values, ["Counting_Unit", "Classifications", "Categories"], "left")
    .fillna(0)
)

for jurisdiction in JURISDICTIONS:
    category_output = category_output.withColumn(
        f"{jurisdiction}_TOTAL",
        F.col(f"{jurisdiction}_Catholic") + F.col(f"{jurisdiction}_Independent")
    )

category_output = (
    category_output
    .withColumn("AUSTRALIA_Catholic", add_columns([F.col(f"{j}_Catholic") for j in JURISDICTIONS]))
    .withColumn("AUSTRALIA_Independent", add_columns([F.col(f"{j}_Independent") for j in JURISDICTIONS]))
    .withColumn("AUSTRALIA_TOTAL", add_columns([F.col(f"{j}_TOTAL") for j in JURISDICTIONS]))
)

DELTA_OUTPUT_COLUMNS = [
    "Counting_Unit", "Classifications", "Categories",
    "NSW_Catholic", "NSW_Independent", "NSW_TOTAL",
    "VIC_Catholic", "VIC_Independent", "VIC_TOTAL",
    "QLD_Catholic", "QLD_Independent", "QLD_TOTAL",
    "SA_Catholic", "SA_Independent", "SA_TOTAL",
    "WA_Catholic", "WA_Independent", "WA_TOTAL",
    "TAS_Catholic", "TAS_Independent", "TAS_TOTAL",
    "NT_Catholic", "NT_Independent", "NT_TOTAL",
    "ACT_Catholic", "ACT_Independent", "ACT_TOTAL",
    "AUSTRALIA_Catholic", "AUSTRALIA_Independent", "AUSTRALIA_TOTAL"
]
MEASURE_COLUMNS = DELTA_OUTPUT_COLUMNS[3:]

total_sort_order = (
    row_order_df
    .filter(F.col("Categories") == "Total")
    .select("_sort_order", "Counting_Unit", "Classifications")
)

total_output = (
    category_output
    .groupBy("Counting_Unit", "Classifications")
    .agg(*[F.sum(c).alias(c) for c in MEASURE_COLUMNS])
    .withColumn("Categories", F.lit("Total"))
    .join(total_sort_order, ["Counting_Unit", "Classifications"], "left")
    .select("_sort_order", *DELTA_OUTPUT_COLUMNS)
)

combined_output = (
    category_output
    .select("_sort_order", *DELTA_OUTPUT_COLUMNS)
    .unionByName(total_output)
)

# =============================================================================
# 9. BUILD AN INDEPENDENT EXPECTED TABLE DIRECTLY FROM TABLE 1 AND TABLE 2
# =============================================================================
# Re-aggregate directly from the classified source tables using Expected_Value.
expected_long_values = union_all([
    create_long_group(table_1_classified, SCHOOL_COUNT_UNIT, "School Level", "School_Level_Category", "school_count", "Expected_Value"),
    create_long_group(table_2_classified, STUDENT_COUNT_UNIT, "Sex", "Sex_Category", "enrolments", "Expected_Value"),
    create_long_group(table_2_classified, STUDENT_COUNT_UNIT, "Indigenous Status", "Indigenous_Category", "enrolments", "Expected_Value"),
    create_long_group(table_2_classified, STUDENT_COUNT_UNIT, "Attendance Status", "Attendance_Category", "enrolments", "Expected_Value"),
    create_long_group(table_2_classified, STUDENT_COUNT_UNIT, "School Level", "Student_School_Level", "enrolments", "Expected_Value"),
    create_long_group(table_2_classified, STUDENT_FTE_UNIT, "Sex", "Sex_Category", "fte", "Expected_Value"),
    create_long_group(table_2_classified, STUDENT_FTE_UNIT, "Indigenous Status", "Indigenous_Category", "fte", "Expected_Value"),
    create_long_group(table_2_classified, STUDENT_FTE_UNIT, "Attendance Status", "Attendance_Category", "fte", "Expected_Value"),
    create_long_group(table_2_classified, STUDENT_FTE_UNIT, "School Level", "Student_School_Level", "fte", "Expected_Value")
])

state_affiliation_scaffold = spark.createDataFrame(
    [(j, a) for j in JURISDICTIONS for a in AFFILIATIONS],
    ["State_Territory", "School_Affiliation"]
)

expected_base_scaffold = (
    row_order_df
    .filter(F.col("Categories") != "Total")
    .select("Counting_Unit", "Classifications", F.col("Categories").alias("Category"))
    .crossJoin(state_affiliation_scaffold)
)

expected_base = (
    expected_base_scaffold
    .join(
        expected_long_values,
        ["Counting_Unit", "Classifications", "Category", "State_Territory", "School_Affiliation"],
        "left"
    )
    .select(
        "Counting_Unit", "Classifications", "Category", "State_Territory", "School_Affiliation",
        F.coalesce(F.col("Expected_Value"), F.lit(0)).alias("Expected_Value")
    )
)

expected_totals = (
    expected_base
    .groupBy("Counting_Unit", "Classifications", "State_Territory", "School_Affiliation")
    .agg(F.sum("Expected_Value").alias("Expected_Value"))
    .withColumn("Category", F.lit("Total"))
    .select("Counting_Unit", "Classifications", "Category", "State_Territory", "School_Affiliation", "Expected_Value")
)
expected_all_rows_long = expected_base.unionByName(expected_totals)

expected_pivot_aggregations = [
    F.sum(
        F.when(
            (F.col("State_Territory") == jurisdiction)
            & (F.col("School_Affiliation") == affiliation),
            F.col("Expected_Value")
        ).otherwise(F.lit(0))
    ).alias(f"{jurisdiction}_{affiliation}")
    for jurisdiction in JURISDICTIONS
    for affiliation in AFFILIATIONS
]

expected_wide = (
    expected_all_rows_long
    .groupBy("Counting_Unit", "Classifications", F.col("Category").alias("Categories"))
    .agg(*expected_pivot_aggregations)
)
for jurisdiction in JURISDICTIONS:
    expected_wide = expected_wide.withColumn(
        f"{jurisdiction}_TOTAL",
        F.col(f"{jurisdiction}_Catholic") + F.col(f"{jurisdiction}_Independent")
    )
expected_wide = (
    expected_wide
    .withColumn("AUSTRALIA_Catholic", add_columns([F.col(f"{j}_Catholic") for j in JURISDICTIONS]))
    .withColumn("AUSTRALIA_Independent", add_columns([F.col(f"{j}_Independent") for j in JURISDICTIONS]))
    .withColumn("AUSTRALIA_TOTAL", add_columns([F.col(f"{j}_TOTAL") for j in JURISDICTIONS]))
)

expected_unformatted_output = (
    row_order_df
    .join(expected_wide, ["Counting_Unit", "Classifications", "Categories"], "left")
    .orderBy("_sort_order")
    .select("_sort_order", *DELTA_OUTPUT_COLUMNS)
)

# =============================================================================
# 10. VALIDATE EVERY UNFORMATTED NUMERIC CELL
# =============================================================================
expected_unformatted_cells = union_all([
    expected_unformatted_output.select(
        "Counting_Unit", "Classifications", "Categories",
        F.lit(c).alias("Output_Column"),
        F.col(c).alias("Expected_Value")
    ) for c in MEASURE_COLUMNS
])
actual_unformatted_cells = union_all([
    combined_output.select(
        "Counting_Unit", "Classifications", "Categories",
        F.lit(c).alias("Output_Column"),
        F.col(c).alias("Actual_Value")
    ) for c in MEASURE_COLUMNS
])

unformatted_cell_validation = add_match_columns(
    expected_unformatted_cells
    .join(
        actual_unformatted_cells,
        ["Counting_Unit", "Classifications", "Categories", "Output_Column"],
        "full"
    )
    .select(
        "Counting_Unit", "Classifications", "Categories", "Output_Column",
        F.coalesce(F.col("Expected_Value"), F.lit(0)).alias("Expected_Value"),
        F.coalesce(F.col("Actual_Value"), F.lit(0)).alias("Actual_Value")
    )
)

# =============================================================================
# 11. STRUCTURAL AND ARITHMETIC CHECKS
# =============================================================================
expected_keys = row_order_df.select("_sort_order", "Counting_Unit", "Classifications", "Categories")
actual_keys = combined_output.select("_sort_order", "Counting_Unit", "Classifications", "Categories")
missing_rows = expected_keys.join(actual_keys, expected_keys.columns, "left_anti")
unexpected_rows = actual_keys.join(expected_keys, expected_keys.columns, "left_anti")
duplicate_rows = actual_keys.groupBy(*actual_keys.columns).count().filter(F.col("count") != 1)

missing_row_count = missing_rows.count()
unexpected_row_count = unexpected_rows.count()
duplicate_row_count = duplicate_rows.count()

expected_combined_columns = ["_sort_order", *DELTA_OUTPUT_COLUMNS]
actual_combined_columns = combined_output.columns
column_order_matches = actual_combined_columns == expected_combined_columns

arithmetic_validation = combined_output
arithmetic_difference_columns = []
for jurisdiction in JURISDICTIONS:
    difference_column = f"{jurisdiction}_TOTAL_DIFFERENCE"
    arithmetic_difference_columns.append(difference_column)
    arithmetic_validation = arithmetic_validation.withColumn(
        difference_column,
        F.col(f"{jurisdiction}_TOTAL")
        - (F.col(f"{jurisdiction}_Catholic") + F.col(f"{jurisdiction}_Independent"))
    )

arithmetic_validation = (
    arithmetic_validation
    .withColumn(
        "AUSTRALIA_Catholic_DIFFERENCE",
        F.col("AUSTRALIA_Catholic") - add_columns([F.col(f"{j}_Catholic") for j in JURISDICTIONS])
    )
    .withColumn(
        "AUSTRALIA_Independent_DIFFERENCE",
        F.col("AUSTRALIA_Independent") - add_columns([F.col(f"{j}_Independent") for j in JURISDICTIONS])
    )
    .withColumn(
        "AUSTRALIA_TOTAL_DIFFERENCE",
        F.col("AUSTRALIA_TOTAL") - add_columns([F.col(f"{j}_TOTAL") for j in JURISDICTIONS])
    )
)
arithmetic_difference_columns += [
    "AUSTRALIA_Catholic_DIFFERENCE",
    "AUSTRALIA_Independent_DIFFERENCE",
    "AUSTRALIA_TOTAL_DIFFERENCE"
]
arithmetic_issue_condition = reduce(
    lambda left, right: left | right,
    [F.abs(F.col(c)) >= F.lit(TOLERANCE) for c in arithmetic_difference_columns]
)
arithmetic_issues = arithmetic_validation.filter(arithmetic_issue_condition)
arithmetic_issue_count = arithmetic_issues.count()

unformatted_cell_count = unformatted_cell_validation.count()
unformatted_failure_count = unformatted_cell_validation.filter(F.col("Status") != "MATCH").count()

if unformatted_failure_count > 0:
    display(unformatted_cell_validation.filter(F.col("Status") != "MATCH"))
if missing_row_count > 0:
    display(missing_rows)
if unexpected_row_count > 0:
    display(unexpected_rows)
if duplicate_row_count > 0:
    display(duplicate_rows)
if arithmetic_issue_count > 0:
    display(arithmetic_issues)

if (
    unformatted_failure_count > 0
    or missing_row_count > 0
    or unexpected_row_count > 0
    or duplicate_row_count > 0
    or not column_order_matches
    or arithmetic_issue_count > 0
):
    raise ValueError("Unformatted output failed complete cell, structure or arithmetic validation.")

# =============================================================================
# 12. FORMAT ACTUAL AND EXPECTED FINAL TABLES
# =============================================================================
formatted_output = apply_output_format(combined_output)
expected_formatted_output = apply_output_format(expected_unformatted_output)

nssc_ngs_edits_2026 = (
    formatted_output
    .orderBy("_sort_order")
    .select(*DELTA_OUTPUT_COLUMNS)
)
expected_final_output = (
    expected_formatted_output
    .orderBy("_sort_order")
    .select(*DELTA_OUTPUT_COLUMNS)
)

# =============================================================================
# 13. VALIDATE EVERY FINAL FORMATTED NUMERIC CELL
# =============================================================================
expected_formatted_cells = union_all([
    expected_final_output.select(
        "Counting_Unit", "Classifications", "Categories",
        F.lit(c).alias("Output_Column"),
        F.col(c).alias("Expected_Value")
    ) for c in MEASURE_COLUMNS
])
actual_formatted_cells = union_all([
    nssc_ngs_edits_2026.select(
        "Counting_Unit", "Classifications", "Categories",
        F.lit(c).alias("Output_Column"),
        F.col(c).alias("Actual_Value")
    ) for c in MEASURE_COLUMNS
])

formatted_cell_validation = add_match_columns(
    expected_formatted_cells
    .join(
        actual_formatted_cells,
        ["Counting_Unit", "Classifications", "Categories", "Output_Column"],
        "full"
    )
    .select(
        "Counting_Unit", "Classifications", "Categories", "Output_Column",
        F.coalesce(F.col("Expected_Value"), F.lit(0)).alias("Expected_Value"),
        F.coalesce(F.col("Actual_Value"), F.lit(0)).alias("Actual_Value")
    )
)

formatted_cell_count = formatted_cell_validation.count()
formatted_failure_count = formatted_cell_validation.filter(F.col("Status") != "MATCH").count()

expected_final_row_count = len(row_order)
actual_final_row_count = nssc_ngs_edits_2026.count()
final_columns_match = nssc_ngs_edits_2026.columns == DELTA_OUTPUT_COLUMNS
invalid_delta_columns = [c for c in nssc_ngs_edits_2026.columns if re.search(r'[ ,;{}()\n\t=]', c)]

expected_labels = expected_final_output.select("Counting_Unit", "Classifications", "Categories")
actual_labels = nssc_ngs_edits_2026.select("Counting_Unit", "Classifications", "Categories")
missing_labels = expected_labels.join(actual_labels, expected_labels.columns, "left_anti")
unexpected_labels = actual_labels.join(expected_labels, expected_labels.columns, "left_anti")
missing_label_count = missing_labels.count()
unexpected_label_count = unexpected_labels.count()

if formatted_failure_count > 0:
    display(formatted_cell_validation.filter(F.col("Status") != "MATCH"))
if missing_label_count > 0:
    display(missing_labels)
if unexpected_label_count > 0:
    display(unexpected_labels)

if (
    formatted_failure_count > 0
    or actual_final_row_count != expected_final_row_count
    or not final_columns_match
    or invalid_delta_columns
    or missing_label_count > 0
    or unexpected_label_count > 0
):
    raise ValueError("Final formatted output failed complete cell-by-cell or structural validation.")

# =============================================================================
# 14. EXPLICIT GRAND-TOTAL RECONCILIATION
# =============================================================================
table_1_school_total = table_1.agg(F.sum("school_count").alias("v")).first()["v"] or 0
table_2_student_total = table_2.agg(F.sum("enrolments").alias("v")).first()["v"] or 0
table_2_fte_total = table_2.agg(F.sum("fte").alias("v")).first()["v"] or 0

grand_total_validation = spark.createDataFrame([
    ("Table 1 School Count", float(table_1_school_total), float(
        combined_output.filter(
            (F.col("Counting_Unit") == SCHOOL_COUNT_UNIT)
            & (F.col("Classifications") == "School Level")
            & (F.col("Categories") == "Total")
        ).select("AUSTRALIA_TOTAL").first()["AUSTRALIA_TOTAL"] or 0
    )),
    ("Table 2 Student Count", float(table_2_student_total), float(
        combined_output.filter(
            (F.col("Counting_Unit") == STUDENT_COUNT_UNIT)
            & (F.col("Classifications") == "Sex")
            & (F.col("Categories") == "Total")
        ).select("AUSTRALIA_TOTAL").first()["AUSTRALIA_TOTAL"] or 0
    )),
    ("Table 2 Student FTE", float(table_2_fte_total), float(
        combined_output.filter(
            (F.col("Counting_Unit") == STUDENT_FTE_UNIT)
            & (F.col("Classifications") == "Sex")
            & (F.col("Categories") == "Total")
        ).select("AUSTRALIA_TOTAL").first()["AUSTRALIA_TOTAL"] or 0
    ))
], ["Measure", "Source_Total", "Output_Total"])

grand_total_validation = (
    grand_total_validation
    .withColumn("Difference", F.col("Output_Total") - F.col("Source_Total"))
    .withColumn("Status", F.when(F.abs(F.col("Difference")) < F.lit(TOLERANCE), "MATCH").otherwise("CHECK"))
)
if grand_total_validation.filter(F.col("Status") != "MATCH").count() > 0:
    display(grand_total_validation)
    raise ValueError("Grand totals do not reconcile with Table 1 and/or Table 2.")

# =============================================================================
# 15. CREATE VALIDATION VIEWS
# =============================================================================
unformatted_cell_validation.createOrReplaceTempView(
    "nssc_ngs_edits_2026_unformatted_cell_validation"
)
formatted_cell_validation.createOrReplaceTempView(
    "nssc_ngs_edits_2026_formatted_cell_validation"
)
expected_final_output.createOrReplaceTempView(
    "nssc_ngs_edits_2026_expected"
)
grand_total_validation.createOrReplaceTempView(
    "nssc_ngs_edits_2026_grand_total_validation"
)
nssc_ngs_edits_2026.createOrReplaceTempView(
    "nssc_ngs_edits_2026"
)

# =============================================================================
# 16. SAVE FINAL DELTA TABLE
# =============================================================================
(
    nssc_ngs_edits_2026
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUTPUT_TABLE)
)

# =============================================================================
# 17. DISPLAY RESULTS
# =============================================================================
validation_summary = spark.createDataFrame([
    ("Unformatted numeric cells", unformatted_cell_count, unformatted_failure_count),
    ("Formatted numeric cells", formatted_cell_count, formatted_failure_count),
    ("Missing output rows", expected_final_row_count, missing_row_count),
    ("Unexpected output rows", expected_final_row_count, unexpected_row_count),
    ("Duplicate output rows", expected_final_row_count, duplicate_row_count),
    ("Arithmetic rows", actual_final_row_count, arithmetic_issue_count),
    ("Missing label rows", expected_final_row_count, missing_label_count),
    ("Unexpected label rows", expected_final_row_count, unexpected_label_count)
], ["Validation_Check", "Values_Checked", "Failures"]).withColumn(
    "Status", F.when(F.col("Failures") == 0, "MATCH").otherwise("CHECK")
)

print(f"Table 1 source: {TABLE_1_SOURCE}")
print(f"Table 2 source: {TABLE_2_SOURCE}")
print(f"Final Edits table: {OUTPUT_TABLE}")
print(f"Table 1 source rows: {table_1_row_count}")
print(f"Table 2 source rows: {table_2_row_count}")
print(f"Final Edits rows: {actual_final_row_count}")
print(f"Unformatted numeric cells checked: {unformatted_cell_count}")
print(f"Formatted numeric cells checked: {formatted_cell_count}")
print("Table 2 Primary grades: 1-7 and 9.")
print("Table 2 Secondary grades: 10-16.")
print("Table 2 grade code 8 is invalid.")

display(validation_summary)
display(grand_total_validation)
display(spark.table(OUTPUT_TABLE))

print("Completed successfully:")
print("- Every numeric cell was independently recalculated from Table 1 or Table 2.")
print("- Every state-affiliation cell, state total and Australia total matched.")
print("- Every category row and total row matched.")
print("- Every final formatted value matched after rounding.")
print("- Every required row, label, column and column position was checked.")
print(f"- Final Delta table created: {OUTPUT_TABLE}")

Table 1 source: pr_dap_efs.business_sdpt_abs.nssc_ngs_table_1_2026
Table 2 source: pr_dap_efs.business_sdpt_abs.nssc_ngs_table_2_2026
Final Edits table: pr_dap_efs.business_sdpt_abs.nssc_ngs_edits_2026
Table 1 source rows: 1009
Table 2 source rows: 5919
Final Edits rows: 35
Unformatted numeric cells checked: 945
Formatted numeric cells checked: 945
Table 2 Primary grades: 1-7 and 9.
Table 2 Secondary grades: 10-16.
Table 2 grade code 8 is invalid.


Validation_Check,Values_Checked,Failures,Status
Unformatted numeric cells,945,0,MATCH
Formatted numeric cells,945,0,MATCH
Missing output rows,35,0,MATCH
Unexpected output rows,35,0,MATCH
Duplicate output rows,35,0,MATCH
Arithmetic rows,35,0,MATCH
Missing label rows,35,0,MATCH
Unexpected label rows,35,0,MATCH


Measure,Source_Total,Output_Total,Difference,Status
Table 1 School Count,2953.0,2953.0,0.0,MATCH
Table 2 Student Count,1573743.0,1573743.0,0.0,MATCH
Table 2 Student FTE,1572983.0,1572983.0,0.0,MATCH


Counting_Unit,Classifications,Categories,NSW_Catholic,NSW_Independent,NSW_TOTAL,VIC_Catholic,VIC_Independent,VIC_TOTAL,QLD_Catholic,QLD_Independent,QLD_TOTAL,SA_Catholic,SA_Independent,SA_TOTAL,WA_Catholic,WA_Independent,WA_TOTAL,TAS_Catholic,TAS_Independent,TAS_TOTAL,NT_Catholic,NT_Independent,NT_TOTAL,ACT_Catholic,ACT_Independent,ACT_TOTAL,AUSTRALIA_Catholic,AUSTRALIA_Independent,AUSTRALIA_TOTAL
School Count,School Level,Primary,418.0,65.0,483.0,389.0,29.0,418.0,196.0,29.0,225.0,60.0,28.0,88.0,109.0,33.0,142.0,24.0,6.0,30.0,8.0,5.0,13.0,24.0,3.0,27.0,1228.0,198.0,1426.0
School Count,School Level,Secondary,127.0,11.0,138.0,84.0,10.0,94.0,57.0,20.0,77.0,9.0,8.0,17.0,20.0,10.0,30.0,5.0,0.0,5.0,4.0,5.0,9.0,6.0,0.0,6.0,312.0,64.0,376.0
School Count,School Level,Combined,36.0,251.0,287.0,14.0,156.0,170.0,47.0,153.0,200.0,27.0,63.0,90.0,29.0,98.0,127.0,8.0,26.0,34.0,5.0,16.0,21.0,2.0,12.0,14.0,168.0,775.0,943.0
School Count,School Level,Special,15.0,64.0,79.0,7.0,33.0,40.0,14.0,36.0,50.0,5.0,6.0,11.0,5.0,15.0,20.0,2.0,3.0,5.0,1.0,0.0,1.0,0.0,2.0,2.0,49.0,159.0,208.0
School Count,School Level,Total,596.0,391.0,987.0,494.0,228.0,722.0,314.0,238.0,552.0,101.0,105.0,206.0,163.0,156.0,319.0,39.0,35.0,74.0,18.0,26.0,44.0,32.0,17.0,49.0,1757.0,1196.0,2953.0
Student Count (including students at special schools),Sex,Male,137361.0,103944.0,241305.0,109588.0,89792.0,199380.0,84850.0,81421.0,166271.0,28061.0,28488.0,56549.0,39357.0,41530.0,80887.0,7843.0,5878.0,13721.0,2765.0,3020.0,5785.0,10526.0,5631.0,16157.0,420351.0,359704.0,780055.0
Student Count (including students at special schools),Sex,Female,138853.0,108427.0,247280.0,110559.0,93883.0,204442.0,81736.0,83590.0,165326.0,28399.0,28747.0,57146.0,39575.0,42925.0,82500.0,8439.0,5926.0,14365.0,2732.0,2989.0,5721.0,9464.0,6354.0,15818.0,419757.0,372841.0,792598.0
Student Count (including students at special schools),Sex,Other,11.0,146.0,157.0,133.0,283.0,416.0,61.0,177.0,238.0,13.0,121.0,134.0,13.0,65.0,78.0,10.0,26.0,36.0,0.0,3.0,3.0,7.0,20.0,27.0,248.0,841.0,1089.0
Student Count (including students at special schools),Sex,Not Stated/Inadequately described,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
Student Count (including students at special schools),Sex,Total,276225.0,212518.0,488743.0,220280.0,183958.0,404238.0,166647.0,165188.0,331835.0,56473.0,57356.0,113829.0,78945.0,84520.0,163465.0,16292.0,11830.0,28122.0,5497.0,6012.0,11509.0,19997.0,12005.0,32002.0,840356.0,733387.0,1573743.0


Completed successfully:
- Every numeric cell was independently recalculated from Table 1 or Table 2.
- Every state-affiliation cell, state total and Australia total matched.
- Every category row and total row matched.
- Every final formatted value matched after rounding.
- Every required row, label, column and column position was checked.
- Final Delta table created: pr_dap_efs.business_sdpt_abs.nssc_ngs_edits_2026
